## Regresion Logistica 

Inicialmente probamos con la regresion logistica, un algoritmo diseñado para algorimtos que son linealmentes separables 

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

def train_logistic_regression(X, y): 
    regularizacion = list(np.logspace(-4, 4, 9))
    rows = []
    for c in regularizacion: 
        accs_train, accs_val, f1s, recalls = [],[],[],[]
        
        kf = KFold(n_splits=5)
        for train_index, val_index in kf.split(X): 
            xtrain, xval = X[train_index], X[val_index]
            ytrain, yval = y[train_index], y[val_index]
            
            scaler = StandardScaler() 
            xtrain, xval = scaler.fit_transform(xtrain), scaler.transform(xval)
            
            model = LogisticRegression(max_iter=1000000, 
                                            tol=1e-6, 
                                            random_state=0,
                                            C=c)

            model.fit(xtrain, ytrain)
            
            predictions_train = model.predict(xtrain)
            predictions_val = model.predict(xval)
            
            accs_train.append(accuracy_score(ytrain, predictions_train))
            accs_val.append(accuracy_score(yval, predictions_val))
            f1s.append(f1_score(yval, predictions_val))
            recalls.append(recall_score(yval, predictions_val))
            

        rows.append({    
            'Regularizacion C': c,
            'media acc train' : np.mean(accs_train),
            'media acc val': np.mean(accs_val), 
            'IC acc val': np.std(accs_val),
            'media f1' : np.mean(f1s),
            'IC f1': np.std(f1s),
            'media recall': np.mean(recalls),
            'IC recall': np.std(recalls)
        }) 

    results = pd.DataFrame(rows)
    return results.sort_values(by='media f1', ascending=False)


In [2]:
# Con los datos sin filtrado 
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned()

train_logistic_regression(X_train, y_train)

,Regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recall
7,1000.0000,0.945250,0.941875,0.003933,0.942355,0.004420,0.948560,0.008656
8,10000.0000,0.945156,0.941750,0.003881,0.942223,0.004385,0.948315,0.008279
6,100.0000,0.945312,0.941750,0.003674,0.942202,0.004101,0.947818,0.008408
5,10.0000,0.945094,0.941625,0.003368,0.942078,0.003727,0.947571,0.008148
4,1.0000,0.944312,0.941375,0.003197,0.941868,0.003419,0.947840,0.008022
3,0.1000,0.941281,0.938875,0.004134,0.939476,0.003909,0.946352,0.007216
2,0.0100,0.933719,0.931875,0.005047,0.932400,0.004937,0.937392,0.009221
1,0.0010,0.923531,0.922250,0.006989,0.922191,0.006890,0.919249,0.011493
0,0.0001,0.902062,0.900625,0.008542,0.898358,0.008813,0.876373,0.015343


In [3]:
# Para los datos filtrados

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned()

train_logistic_regression(X_train, y_train)

,Regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recall
5,10.0000,0.943937,0.941500,0.006029,0.941815,0.005925,0.948999,0.008522
6,100.0000,0.943937,0.941500,0.006029,0.941799,0.005950,0.948749,0.008797
8,10000.0000,0.943906,0.941500,0.006029,0.941799,0.005950,0.948749,0.008797
7,1000.0000,0.943906,0.941500,0.006029,0.941799,0.005950,0.948749,0.008797
4,1.0000,0.943000,0.941375,0.005908,0.941680,0.005967,0.948991,0.009562
3,0.1000,0.939344,0.939125,0.006933,0.939361,0.007137,0.945732,0.010054
2,0.0100,0.934250,0.933750,0.006098,0.933906,0.006532,0.938965,0.010034
1,0.0010,0.923563,0.922375,0.007608,0.921631,0.008715,0.916393,0.013780
0,0.0001,0.898656,0.897125,0.007980,0.892718,0.009182,0.858771,0.019476


Se obtienen resultados considerablemente buenos cuando se itera con parametro de regularización $c=100$ 

Aunque es muy probable que según los resultados obtenidos el problema de naturaleza lineal, probaremos con diferentes enfoques más allá de una simple linea

Si usamos como nuestra métrica principal el `f1_score` vemos que nuestro mejor modelo es el que usa el kernel `rbf` y valor $c=10$

## Extraccion de caracteristicas Lineal (PCA)



Tenemos que los mejores modelos son cuando `C = 1000` y `c = 100` ahora aplicaremos reduccion de dimencion para estos modelos 

In [2]:
from libs.data_configuration import  get_filtered_data_partitioned
from sklearn.decomposition import PCA
from libs.linear_reduction import train_models

X_train, X_test, y_train, y_test = get_filtered_data_partitioned()

print(f"Iniciamos con un conjunto de {X_train.shape[1]} caracteristicas")

models =[ 
    LogisticRegression(max_iter=1000000, 
                        tol=1e-6, 
                        random_state=0,
                        C=1000),
    LogisticRegression(max_iter=1000000, 
                            tol=1e-6, 
                            random_state=0,
                            C=100)
]

scaler = StandardScaler() 
xtrain, xtest = scaler.fit_transform(X_train), scaler.transform(X_test)

pca = PCA(n_components=0.95) # conservamos el 95% de la varianza 

xtrain, xtest = pca.fit_transform(xtrain), pca.transform(xtest)

print(f"Terminamos con un conunto de {xtrain.shape[1]} caracteristicas despues de la reduccion con PCA")

train_models(xtrain, xtest, y_train, y_test, models, ['C'])


Iniciamos con un conjunto de 41 caracteristicas
Terminamos con un conunto de 32 caracteristicas despues de la reduccion con PCA


,C,acc train,acc test,f1_score,recall
0,1000,0.93375,0.9295,0.928171,0.937243
1,100,0.93375,0.9295,0.928171,0.937243
